# 03 · Join Sofascore + Capology — Germany Bundesliga 20/21

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2020/21 de Bundesliga alemana**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_germany_2021.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_germany_2021.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  496 jugadores | 116 columnas
Capology:   552 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   1 fc koln
   1 fc union berlin
   1 fsv mainz 05
   bayer 04 leverkusen
   borussia m gladbach
   fc augsburg
   fc bayern munchen
   fc schalke 04
   hertha bsc
   holstein kiel
   rb leipzig
   sc freiburg
   sv werder bremen
   tsg hoffenheim
   vfb stuttgart
   vfl wolfsburg

En Capology pero no en Sofascore:
   augsburg
   bayer leverkusen
   bayern munich
   freiburg
   hertha berlin
   hoffenheim
   koln
   leipzig
   mainz
   monchengladbach
   schalke 04
   stuttgart
   union berlin
   werder bremen
   wolfsburg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'augsburg':'fc augsburg',
            'bayer leverkusen':'bayer 04 leverkusen',
            'bayern munich':'fc bayern munchen',
            'freiburg':'sc freiburg',
            'hertha berlin':'hertha bsc',
            'hoffenheim':'tsg hoffenheim',
            'koln':'1 fc koln',
            'leipzig':'rb leipzig',
            'mainz':'1 fsv mainz 05',
            'monchengladbach':'borussia m gladbach',
            'schalke 04':'fc schalke 04',
            'stuttgart':'vfb stuttgart',
            'union berlin':'1 fc union berlin',
            'werder bremen':'sv werder bremen',
            'wolfsburg':'vfl wolfsburg'                          
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 445/496 (89.7%)
Sin emparejar: 51


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          12
Revisión media    (0.75 ≤ score < 0.90):   9
Revisión estricta (0.50 ≤ score < 0.75):   17
Revisión muy est. (score < 0.50):           12


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
29,Alfreð Finnbogason,FC Augsburg,alfred finnbogason,0.971
7,Alexander Sørloth,RB Leipzig,alexander sorloth,0.970
19,Dimitris Limnios,1. FC Köln,dimitrios limnios,0.970
35,Frederik Sørensen,1. FC Köln,frederik sorensen,0.970
8,Frederik Rønnow,FC Schalke 04,frederik ronnow,0.966
5,Rafał Gikiewicz,FC Augsburg,rafal gikiewicz,0.966
16,Łukasz Piszczek,Borussia Dortmund,lukasz piszczek,0.966
14,Bartosz Białek,VfL Wolfsburg,bartosz bialek,0.963
13,Levin Öztunalı,1. FSV Mainz 05,levin oztunali,0.963
0,Elias Boerdner,Eintracht Frankfurt,elias bordner,0.963


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
43,Noah Joel Sarenren Bazee,FC Augsburg,noah sarenren bazee,0.884
32,Jan Luca Schuler,FC Schalke 04,luca schuler,0.857
15,Anderson-Lenda Lucoqui,Arminia Bielefeld,anderson lucoqui,0.842
22,Jóan Edmundsson,Arminia Bielefeld,joan simun edmundsson,0.833
48,Matondo-Merveille Papela,1. FSV Mainz 05,merveille papela,0.800
9,Leandro Barreiro,1. FSV Mainz 05,leandro barreiro martins,0.800
4,Stefan Ortega,Arminia Bielefeld,stefan ortega moreno,0.788
12,Mateu Morey,Borussia Dortmund,mateu morey bauza,0.786
18,Pierre Kunde,1. FSV Mainz 05,pierre kunde malong,0.774


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 9 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
6,John Brooks,VfL Wolfsburg,john anthony brooks,0.733
28,Johannes Eggestein,SV Werder Bremen,maximilian eggestein,0.632
21,Victor Sá,VfL Wolfsburg,joao victor,0.600
17,Joshua Zirkzee,FC Bayern München,joshua kimmich,0.571
49,Mikail Maden,FC Schalke 04,michael langer,0.538
44,Jonas Dirkner,Hertha BSC,lukas klunter,0.538
31,Philipp Bargfrede,SV Werder Bremen,ilia gruev,0.519
24,Nishan Burkart,SC Freiburg,janik haberer,0.519
27,Florian Flick,FC Schalke 04,kilian ludewig,0.519
46,Marten Winkler,Hertha BSC,omar alderete,0.519


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['john brooks',
                    'victor sa'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 2


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
3,Blendi Idrizi,FC Schalke 04,benito raman,0.480
10,Iago Borduchi,FC Augsburg,dion berisha,0.480
34,Dennis Borkowski,RB Leipzig,dani olmo,0.480
47,Vasilios Pavlidis,FC Schalke 04,goncalo paciencia,0.471
37,Emrehan Gedikli,Bayer 04 Leverkusen,demarai gray,0.444
40,Anthony Modeste,1. FC Köln,ron robert zieler,0.438
26,Tolu Arokodare,1. FC Köln,timo horn,0.435
25,Jonas Michelbrink,Hertha BSC,javairo dilrosun,0.424
41,Eren Dinkçi,SV Werder Bremen,romano schmid,0.417
38,Jimmy Kaparos,FC Schalke 04,timo becker,0.417


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = ['silas'

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 1


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 469/496 (94.6%)
Sin salario:     27


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 27


,player,team,minutesPlayed,appearances,goals,assists
0,Anthony Modeste,1. FC Köln,215,8,0,0
1,Tolu Arokodare,1. FC Köln,114,10,0,0
2,Emrehan Gedikli,Bayer 04 Leverkusen,24,1,0,0
3,Iago Borduchi,FC Augsburg,1564,18,1,1
4,Lukas Petkov,FC Augsburg,3,1,0,0
5,Joshua Zirkzee,FC Bayern München,92,3,0,0
6,Josip Stanišić,FC Bayern München,90,1,0,0
7,Christopher Scott,FC Bayern München,32,2,0,0
8,Michaël Cuisance,FC Bayern München,18,1,0,0
9,Mehmet Aydın,FC Schalke 04,501,6,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  1. FC Köln  —  SF sin salario:


,player,minutesPlayed
0,Anthony Modeste,215
1,Tolu Arokodare,114


  CG plantilla completa:


,player,player_norm
0,Benno Schmitz,benno schmitz
1,Christian Clemens,christian clemens
2,Dimitrios Limnios,dimitrios limnios
3,Dominick Drexler,dominick drexler
4,Ellyes Skhiri,ellyes skhiri
5,Elvis Rexhbecaj,elvis rexhbecaj
6,Emmanuel Dennis,emmanuel dennis
7,Florian Kainz,florian kainz
8,Frederik Sörensen,frederik sorensen
9,Ismail Jakobs,ismail jakobs



  Bayer 04 Leverkusen  —  SF sin salario:


...

,player,player_norm
0,Christian Groß,christian gro
1,Davie Selke,davie selke
2,Eduardo Dos Santos Haesler,eduardo dos santos haesler
3,Felix Agu,felix agu
4,Ilia Gruev,ilia gruev
5,Jean Manuel Mbom,jean manuel mbom
6,Jiri Pavlenka,jiri pavlenka
7,Josh Sargent,josh sargent
8,Kevin Möhwald,kevin mohwald
9,Leonardo Bittencourt,leonardo bittencourt



  VfB Stuttgart  —  SF sin salario:


,player,minutesPlayed
0,Mohamed Sankoh,16


  CG plantilla completa:


,player,player_norm
0,Aílton,ailton
1,Antonis Aidonis,antonis aidonis
2,Atakan Karazor,atakan karazor
3,Borna Sosa,borna sosa
4,Clinton Mola,clinton mola
5,Daniel Didavi,daniel didavi
6,Darko Churlinov,darko churlinov
7,Erik Thommy,erik thommy
8,Fabian Bredlow,fabian bredlow
9,Gonzalo Castro,gonzalo castro


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('iago borduchi','fc augsburg'):('iago',('fc augsburg'))
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: iago borduchi (fc augsburg) → iago (fc augsburg)

Tras matches manuales: 470/496 (94.8%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_germany_2021.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_germany_2021.csv
   Jugadores totales:  496
   Con salario:        470
   Sin salario (NaN):  26
   Columnas:           121
